In [1]:
import pandas as pd
df = pd.read_csv("data.csv")
df

,id,exercise_name,type_of_activity,type_of_equipment,body_part,type,muscle_groups_activated,instructions
0,0,Push-Ups,Strength,Bodyweight,Upper Body,Push,"Pectorals, Triceps, Deltoids",Start in a high plank position with your hands...
1,1,Squats,Strength,Bodyweight,Lower Body,Push,"Quadriceps, Glutes, Hamstrings",Stand with feet shoulder-width apart. Lower yo...
2,2,Plank,Strength/Mobility,Bodyweight,Core,Hold,"Rectus Abdominis, Transverse Abdominis",Start in a forearm plank position with your el...
3,3,Deadlift,Strength,Barbell,Lower Body,Pull,"Glutes, Hamstrings, Lower Back","Stand with feet hip-width apart, barbell in fr..."
4,4,Bicep Curls,Strength,Dumbbells,Upper Body,Pull,"Biceps, Forearms","Stand with a dumbbell in each hand, arms fully..."
...,...,...,...,...,...,...,...,...
202,202,Incline Dumbbell Row,Strength,Dumbbells,Upper Body,Pull,"Latissimus Dorsi, Biceps",Lie face down on an incline bench with a dumbb...
203,203,Machine Lat Pulldown,Strength,Machine,Upper Body,Pull,"Latissimus Dorsi, Biceps",Sit at a lat pulldown machine with a wide grip...
204,204,One-Arm Cable Row,Strength,Cable Machine,Upper Body,Pull,"Latissimus Dorsi, Biceps",Stand facing a cable machine with the handle a...
205,205,Face Pull,Strength,Cable Machine,Upper Body,Pull,"Rear Deltoids, Trapezius, Rhomboids",Attach a rope handle to a high cable pulley. P...


In [2]:
documents = df.to_dict(orient='records')

In [3]:
import minsearch
index = minsearch.Index(
    text_fields=['exercise_name', 'type_of_activity', 'type_of_equipment', 'body_part',
       'type', 'muscle_groups_activated', 'instructions'],
    keyword_fields=['id']
)
index.fit(documents)

/Users/swetha/Desktop/RAG fitness assistant/notebooks/minsearch.py:10: UserWarning: Now minsearch is available at pypi. Run 'pip install minsearch' or 'uv add minsearch' to use it. Remove the downloaded file ('rm minsearch.py') and re-install it.
  warnings.warn(


In [4]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [5]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small"
)

def llm(prompt, model=None):
    result = generator(prompt, max_length=256)
    return result[0]["generated_text"]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'Deepseek

In [6]:
def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=10
    )

    return results

In [7]:
prompt_template = """
You're a fitness insrtuctor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

entry_template = """
exercise_name: {exercise_name}
type_of_activity: {type_of_activity}
type_of_equipment: {type_of_equipment}
body_part: {body_part}
type: {type}
muscle_groups_activated: {muscle_groups_activated}
instructions: {instructions}
""".strip()

def build_prompt(query, search_results):
    context = ""
    
    for doc in search_results:
        context = context + entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [8]:
def rag(query, model='gpt-4o-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    #print(prompt)
    answer = llm(prompt, model=model)
    return answer

In [9]:
question = 'Is the Lat Pulldown considered a strength training activity, and if so, why?'
answer = rag(question)
print(answer)

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You're a fitness insrtuctor. Answer the QUESTION based on the CONTEXT from our exercises database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: Is the Lat Pulldown considered a strength training activity, and if so, why?

CONTEXT:
exercise_name: Lat Pulldown
type_of_activity: Strength
type_of_equipment: Machine
body_part: Upper Body
type: Pull
muscle_groups_activated: Latissimus Dorsi, Biceps
instructions: Sit at the lat pulldown machine, grasp the bar with a wide grip, and pull it down to your chest. Slowly return to the starting position.

exercise_name: Cable Lat Pulldown
type_of_activity: Strength
type_of_equipment: Cable Machine
body_part: Upper Body
type: Pull
muscle_groups_activated: Latissimus Dorsi, Biceps
instructions: Sit at a lat pulldown machine, grasp the bar with a wide grip, and pull it down to your chest. Slowly return to the starting position.

exercise_name: Machine Lat Pulldown
type_of_activity: Strength
type_of_equipment: Machine
body

In [10]:
prompt_template = """
You are generating questions for a fitness assistant dataset.

Generate ONE clear and specific question about the following exercise.

Rules:
- MUST include the exercise name
- MUST be about muscles, body part, equipment, or how to perform it
- MUST be realistic and meaningful
- DO NOT ask vague questions like "best way to do this"
- DO NOT ask nonsense or unrelated questions

Exercise: {exercise_name}
Type: {type_of_activity}
Equipment: {type_of_equipment}
Body part: {body_part}
Muscles: {muscle_groups_activated}

Question:
""".strip()

In [11]:
prompt = prompt_template.format(**documents[0])

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def llm(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [13]:
def is_valid(q, doc):
    q_lower = q.lower()
    name = doc['exercise_name'].lower()

    return (
        len(q) > 20 and
        name in q_lower and
        "best way" not in q_lower and
        "how many" not in q_lower and
        "name of" not in q_lower and
        "ingredient" not in q_lower and
        "book" not in q_lower
    )

In [14]:
def generate_questions(doc):
    questions = set()

    for _ in range(20):  # generate more, pick best
        q = llm(prompt_template.format(**doc)).strip()

        if is_valid(q, doc):
            questions.add(q)

        if len(questions) == 5:
            break

    # fallback if model struggles
    if len(questions) < 5:
        fallback = [
            f"What muscles are used in {doc['exercise_name']}?",
            f"What body part does {doc['exercise_name']} target?",
            f"What equipment is required for {doc['exercise_name']}?",
            f"How do you perform {doc['exercise_name']} correctly?",
            f"What type of exercise is {doc['exercise_name']}?"
        ]
        for f in fallback:
            questions.add(f)

    return list(questions)[:5]

In [15]:
from tqdm.auto import tqdm

In [16]:
results = {}

for doc in tqdm(documents):
    doc_id = doc['id']
    
    questions = generate_questions(doc)
    
    # Debug (IMPORTANT)
    if not questions:
        print(f"Empty output for doc {doc_id}")
    
    results[doc_id] = questions

  0%|          | 0/207 [00:00<?, ?it/s]

In [17]:
final_results = []

In [18]:
for doc_id, questions in results.items():
    for q in questions:
       final_results.append((doc_id, q))

In [19]:
questions = generate_questions(doc)
results[doc_id] = questions

In [22]:
final_results

[(0, 'What type of equipment does Push-Ups use?'),
 (0, 'What type of equipment does push-ups have?'),
 (0, 'What type of equipment is used for push-ups?'),
 (0, 'What equipment is used to do push-ups?'),
 (0, 'What type of equipment does push-ups come in?'),
 (1, 'What exercise is used to perform squats?'),
 (1, 'What type of equipment does the squats use?'),
 (1, 'What type of exercises are used in squats?'),
 (1, 'What is the type of equipment for squats?'),
 (1, 'What is the type of exercise that the squats are?'),
 (2, 'What is the weight of a plank?'),
 (2, 'What type of equipment does the plank type have?'),
 (2, 'What is the number of muscles in a plank?'),
 (2, 'What is the shortest amount of time the planks should be?'),
 (2, 'What is the first word in a plank?'),
 (3, 'What kind of exercises does the Deadlift Type have?'),
 (3, 'What type of equipment does the deadlift use?'),
 (3, 'What kind of equipment does a deadlift use?'),
 (3, 'What kind of exercises does the deadlift

In [23]:
df_results = pd.DataFrame(final_results, columns=['id', 'question'])

In [24]:
df_results.to_csv('ground-truth-retrieval.csv', index=False)

In [25]:
!head ground-truth-retrieval.csv

id,question
0,What type of equipment does Push-Ups use?
0,What type of equipment does push-ups have?
0,What type of equipment is used for push-ups?
0,What equipment is used to do push-ups?
0,What type of equipment does push-ups come in?
1,What exercise is used to perform squats?
1,What type of equipment does the squats use?
1,What type of exercises are used in squats?
1,What is the type of equipment for squats?
